# NHRT Further Steps — Kaggle T4 (12h limit)

**v3 — Aggressively optimized for ACT's 1.76s/step speed**

| Step | What | Est. Time |
|------|------|----------|
| 1 | Seed Variance: ACT×3 seeds + DR×1 seed | ~2.5h |
| 2 | Scaled DPO: SFT + rejection sampling + DPO | ~1h |
| 3 | Generalization: Sudoku→Maze transfer | ~40m |
| 4 | Layer Ablation: 3 configs | ~2h |
| | **Total** | **~6h** |

In [ ]:
%%time
!git clone https://github.com/jagan25-mj/NHRT.git /kaggle/working/NHRT 2>/dev/null || echo "Already cloned"
%cd /kaggle/working/NHRT
!pip install einops tqdm pydantic wandb omegaconf hydra-core huggingface_hub coolname --quiet 2>&1 | tail -1

import os, json, random, time, gc, math, copy
os.environ["WANDB_MODE"] = "disabled"
os.environ["DISABLE_COMPILE"] = "1"
os.environ["OMP_NUM_THREADS"] = "4"

import torch
torch.backends.cuda.enable_flash_sdp(False)
torch.backends.cuda.enable_mem_efficient_sdp(False)
import torch.nn.functional as F
from torch.optim import AdamW
from torch.utils.data import DataLoader
import numpy as np
from tqdm import tqdm

print(f"GPU: {torch.cuda.get_device_name(0)} | {torch.cuda.get_device_properties(0).total_mem/1e9:.1f}GB")

## Generate 6×6 Sudoku (n_test=300)

In [ ]:
from dataset.common import PuzzleDatasetMetadata

def is_valid(b,r,c,v):
    if v in b[r]: return False
    if v in b[:,c]: return False
    br,bc=2*(r//2),3*(c//3)
    return v not in b[br:br+2,bc:bc+3]

def solve(b):
    for r in range(6):
        for c in range(6):
            if b[r,c]==0:
                ns=list(range(1,7)); random.shuffle(ns)
                for v in ns:
                    if is_valid(b,r,c,v):
                        b[r,c]=v
                        if solve(b): return True
                        b[r,c]=0
                return False
    return True

def save_ds(name, out, n, holes=20):
    res={k:[] for k in ["inputs","labels","puzzle_identifiers","puzzle_indices","group_indices"]}
    res["puzzle_indices"].append(0); res["group_indices"].append(0)
    for i in tqdm(range(n), desc=name):
        b=np.zeros((6,6),dtype=int); solve(b); sol=b.copy(); puz=sol.copy()
        cs=[(r,c) for r in range(6) for c in range(6)]; random.shuffle(cs)
        for r,c in cs[:holes]: puz[r,c]=0
        res["inputs"].append(puz.flatten()+1); res["labels"].append(sol.flatten()+1)
        res["puzzle_indices"].append(i+1); res["puzzle_identifiers"].append(0); res["group_indices"].append(i+1)
    res={k:np.array(v,dtype=np.int32) for k,v in res.items()}
    meta=PuzzleDatasetMetadata(seq_len=36,vocab_size=8,pad_id=0,ignore_label_id=0,
        blank_identifier_id=1,num_puzzle_identifiers=1,total_groups=n,mean_puzzle_examples=1,sets=["all"])
    d=os.path.join(out,name); os.makedirs(d,exist_ok=True)
    with open(os.path.join(d,"dataset.json"),"w") as f: json.dump(meta.model_dump(),f)
    for k,v in res.items(): np.save(os.path.join(d,f"all__{k}.npy"),v)
    with open(os.path.join(out,"identifiers.json"),"w") as f: json.dump(["<blank>"],f)

save_ds("train","data/sudoku-6x6",1000)
save_ds("test","data/sudoku-6x6",300)
print("Dataset ready!")

## Core Helpers

In [ ]:
from puzzle_dataset import PuzzleDataset, PuzzleDatasetConfig
from utils.functions import load_model_class
from models.sparse_embedding import CastedSparseEmbeddingSignSGD_Distributed
from models.losses import IGNORE_LABEL_ID

def cosine_lr(step, base, warmup, total):
    if step < warmup: return base*step/max(1,warmup)
    p=(step-warmup)/max(1,total-warmup)
    return base*max(0,0.5*(1+math.cos(math.pi*p)))

def mkload(path, bs, seed, ep=1):
    tr=PuzzleDataset(PuzzleDatasetConfig(seed=seed,dataset_path=path,global_batch_size=bs,
        test_set_mode=False,epochs_per_iter=ep,rank=0,num_replicas=1),split="train")
    te=PuzzleDataset(PuzzleDatasetConfig(seed=seed,dataset_path=path,global_batch_size=bs,
        test_set_mode=True,epochs_per_iter=1,rank=0,num_replicas=1),split="test")
    tl=DataLoader(tr,batch_size=None,num_workers=1,prefetch_factor=4,pin_memory=True,persistent_workers=True)
    el=DataLoader(te,batch_size=None,num_workers=1,prefetch_factor=4,pin_memory=True,persistent_workers=True)
    return tl,el,tr.metadata

def mkmodel(arch,loss,lt,over,meta,bs,seed):
    torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)
    cfg=dict(batch_size=bs,vocab_size=meta.vocab_size,seq_len=meta.seq_len,
             num_puzzle_identifiers=max(meta.num_puzzle_identifiers,meta.blank_identifier_id+1),
             causal=False,**over)
    with torch.device("cuda"):
        m=load_model_class(loss)(load_model_class(arch)(cfg),loss_type=lt)
    po=CastedSparseEmbeddingSignSGD_Distributed(m.model.puzzle_emb.buffers(),lr=0,weight_decay=0.1,world_size=1)
    mo=AdamW(m.parameters(),lr=0,weight_decay=0.1,betas=(0.9,0.95))
    print(f"  Params: {sum(p.numel() for p in m.parameters()):,}")
    return m,[po,mo]

def do_train(m, opts, loader, epochs, bs, lr, meta, plr=1e-2, warmup=50):
    total=int(epochs*meta.total_groups*meta.mean_puzzle_examples/bs)
    m.train(); step=0; carry=None; losses=[]
    pbar=tqdm(total=total,desc="Training")
    for _,batch,gbs in loader:
        step+=1
        if step>total: break
        batch={k:v.cuda() for k,v in batch.items()}
        if carry is None:
            with torch.device("cuda"): carry=m.initial_carry(batch)
        carry,loss,_,_,_=m(carry=carry,batch=batch,return_keys=[])
        ((1/gbs)*loss).backward()
        for o,l in zip(opts,[cosine_lr(step,plr,warmup,total),cosine_lr(step,lr,warmup,total)]):
            for pg in o.param_groups: pg['lr']=l
            o.step(); o.zero_grad()
        losses.append(loss.item()/gbs)
        pbar.update(1)
        if step%200==0: pbar.set_postfix(loss=f"{np.mean(losses[-50:]):.4f}")
    pbar.close()
    print(f"  Loss: {np.mean(losses[-50:]):.4f} ({step} steps)")
    return m

def do_eval(m, loader, bs):
    m.eval(); tc=tt=te=tp=0
    with torch.no_grad():
        for _,batch,_ in loader:
            batch={k:v.cuda() for k,v in batch.items()}
            with torch.device("cuda"): carry=m.initial_carry(batch)
            while True:
                carry,_,_,preds,done=m(carry=carry,batch=batch,return_keys=["logits"])
                if done: break
            lab=carry.current_data["labels"]; logits=preds.get("logits")
            if logits is not None:
                pred=torch.argmax(logits,dim=-1); mask=lab!=IGNORE_LABEL_ID
                correct=(pred==lab)&mask; tc+=correct.sum().item(); tt+=mask.sum().item()
                for i in range(lab.shape[0]):
                    mm=mask[i]
                    if mm.sum()>0:
                        tp+=1
                        if correct[i].sum()==mm.sum(): te+=1
    return 100*tc/max(tt,1), 100*te/max(tp,1), tp

def clean(*a):
    for x in a: del x
    gc.collect(); torch.cuda.empty_cache()

# ── Config ──
BASE=dict(hidden_size=512,num_heads=8,expansion=4,puzzle_emb_ndim=512,pos_encodings="rope")
DATA="data/sudoku-6x6"
BS=128
EP=150       # 150 epochs → ~1172 steps → ~34min/run (DR) or ~20min with halt=8
LR=3e-4     # Higher LR for fewer steps
SEEDS=[42,123,2024]

print(f"Steps/run: {150*1000//128} | Est ~20-34 min/run depending on halt_max")
print("Helpers loaded.")

---
# STEP 1: Seed Variance

Train DeepReasoner (seed 2024, the missing one) and ACT (all 3 seeds).

**Key change**: `halt_max_steps` reduced from 16→8 for ACT. This cuts training time in half while still allowing adaptive halting.

In [ ]:
CFGS = {
    "DeepReasoner": {
        "a":"hrm.hrm_deep_v1@HierarchicalReasoningModel_DeepV1",
        "l":"losses@ACTLossHead", "lt":"stablemax_cross_entropy",
        "o":{**BASE, "halt_exploration_prob":0.0, "halt_max_steps":3,
             "H_cycles":2,"L_cycles":2,"H_layers":4,"L_layers":4},
        "seeds": [2024],  # only the missing one
    },
    "ACT": {
        "a":"hrm.hrm_act_v1@HierarchicalReasoningModel_ACTV1",
        "l":"losses@ACTLossHead", "lt":"stablemax_cross_entropy",
        "o":{**BASE, "halt_exploration_prob":0.1, "halt_max_steps":8,  # REDUCED from 16→8
             "H_cycles":2,"L_cycles":2,"H_layers":4,"L_layers":4},
        "seeds": [42, 123, 2024],  # all 3
    },
}

t0=time.time()
seed_results={}

for name,cfg in CFGS.items():
    seed_results[name]={}
    for seed in cfg["seeds"]:
        print(f"\n{'='*50}\n  {name} | Seed {seed}\n{'='*50}")
        tl,el,meta=mkload(DATA,BS,seed,train_epochs=EP)
        m,opts=mkmodel(cfg["a"],cfg["l"],cfg["lt"],cfg["o"],meta,BS,seed)
        m=do_train(m,opts,tl,EP,BS,LR,meta)
        c,e,n=do_eval(m,el,BS)
        seed_results[name][seed]=(c,e)
        print(f"  ✅ Cell={c:.2f}% | Exact={e:.2f}% (n={n})")
        d=f"/kaggle/working/ckpts/seeds/{name}"; os.makedirs(d,exist_ok=True)
        torch.save(m.state_dict(),f"{d}/s{seed}.pt")
        del m,opts,tl,el; clean()

print(f"\n⏱️ Step 1: {(time.time()-t0)/60:.1f} min")

# Summary with prior results
PRIOR={"HRB":{42:10.33,123:11.00,2024:13.00},"DeepReasoner":{42:13.67,123:12.33},"Baseline":{42:6.33}}
print(f"\n{'Arch':<16} {'S42':>8} {'S123':>8} {'S2024':>8} {'Mean±Std':>14}")
print("-"*58)
print(f"{'Baseline':<16} {'6.33%':>8} {'6.33%':>8} {'6.33%':>8} {'6.33±0.00':>14}")
h=PRIOR["HRB"]; print(f"{'HRB(prior)':<16} {h[42]:>7.2f}% {h[123]:>7.2f}% {h[2024]:>7.2f}% {np.mean(list(h.values())):>6.2f}±{np.std(list(h.values())):.2f}")
# DR
dr={**PRIOR["DeepReasoner"],**{s:e for s,(c,e) in seed_results.get("DeepReasoner",{}).items()}}
v=[dr[s] for s in SEEDS if s in dr]
print(f"{'DeepReasoner':<16}"+"".join(f" {dr.get(s,0):>7.2f}%" for s in SEEDS)+f" {np.mean(v):>6.2f}±{np.std(v):.2f}")
# ACT
if "ACT" in seed_results:
    av=[seed_results["ACT"][s][1] for s in SEEDS if s in seed_results["ACT"]]
    print(f"{'ACT(new)':<16}"+"".join(f" {seed_results['ACT'].get(s,(0,0))[1]:>7.2f}%" for s in SEEDS)+f" {np.mean(av):>6.2f}±{np.std(av):.2f}")

---
# STEP 2: Scaled DPO
SFT baseline → STaR rejection sampling → DPO with train/val split

In [ ]:
# 2a: SFT baseline
t0=time.time()
cfg=CFGS["ACT"]
tl,el,meta=mkload(DATA,BS,42,train_epochs=EP)
sft,so=mkmodel(cfg["a"],cfg["l"],cfg["lt"],cfg["o"],meta,BS,42)
sft=do_train(sft,so,tl,EP,BS,LR,meta)
sc,se,sn=do_eval(sft,el,BS)
print(f"✅ SFT: Cell={sc:.2f}% | Exact={se:.2f}%")
os.makedirs("/kaggle/working/ckpts/dpo",exist_ok=True)
torch.save(sft.state_dict(),"/kaggle/working/ckpts/dpo/sft.pt")
del tl; gc.collect(); torch.cuda.empty_cache()
print(f"⏱️ SFT: {(time.time()-t0)/60:.1f} min")

In [ ]:
# 2b: Generate preference pairs
t0=time.time()
sft.eval()
pl,_,_=mkload(DATA,1,42,ep=1)
chosen=[]; rejected=[]; pc=0; px=0

with torch.no_grad():
    for _,batch,_ in tqdm(pl,desc="Sampling",total=300):
        px+=1
        if px>300: break
        batch={k:v.cuda() for k,v in batch.items()}
        lab=batch["labels"].clone()
        clog=[]; wlog=[]
        for si in range(4):
            with torch.device("cuda"): carry=sft.initial_carry(batch)
            if si>0:
                carry.inner_carry.z_H+=torch.randn_like(carry.inner_carry.z_H)*0.02
                carry.inner_carry.z_L+=torch.randn_like(carry.inner_carry.z_L)*0.02
            while True:
                carry,_,_,preds,done=sft(carry=carry,batch=batch,return_keys=["logits"])
                if done: break
            logits=preds["logits"]; pred=torch.argmax(logits,dim=-1)
            mask=lab!=IGNORE_LABEL_ID
            ok=((pred==lab)&mask).sum()==mask.sum()
            (clog if ok else wlog).append(logits.cpu())
        if clog and wlog:
            chosen.append((batch["inputs"].cpu(),batch["labels"].cpu(),clog[0]))
            rejected.append((batch["inputs"].cpu(),batch["labels"].cpu(),wlog[0]))
            pc+=1

yld=100*pc/max(px,1)
print(f"✅ {pc} pairs ({yld:.1f}% yield)")
idx=list(range(pc)); random.shuffle(idx)
sp=int(0.8*pc); ti,vi=idx[:sp],idx[sp:]
print(f"  Train:{len(ti)} Val:{len(vi)}")
print(f"⏱️ {(time.time()-t0)/60:.1f} min")

In [ ]:
# 2c: DPO
t0=time.time()
dm=copy.deepcopy(sft); rm=copy.deepcopy(sft)
for p in rm.parameters(): p.requires_grad=False
rm.eval()
dopt=AdamW([p for p in dm.parameters() if p.requires_grad],lr=5e-5,weight_decay=0.01)

def lp(logits_c,lab):
    l=logits_c.cuda().float(); mask=lab.cuda()!=IGNORE_LABEL_ID
    lps=F.log_softmax(l,dim=-1)
    return (torch.gather(lps,-1,lab.cuda().clamp(min=0).unsqueeze(-1)).squeeze(-1)*mask.float()).sum(-1)

BETA=0.1
for ep in range(3):
    dm.train(); ls=[]; pa=[]
    for i in ti:
        _,lab,cl=chosen[i]; _,_,rl=rejected[i]
        pic=lp(cl,lab); pir=lp(rl,lab)
        with torch.no_grad(): rc=lp(cl,lab); rr=lp(rl,lab)
        d=BETA*((pic-pir)-(rc-rr)); loss=-F.logsigmoid(d).mean()
        loss.backward(); torch.nn.utils.clip_grad_norm_(dm.parameters(),1.0)
        dopt.step(); dopt.zero_grad()
        ls.append(loss.item()); pa.append((d>0).float().mean().item())
    dm.eval(); va=[]
    with torch.no_grad():
        for i in vi:
            _,lab,cl=chosen[i]; _,_,rl=rejected[i]
            d=BETA*((lp(cl,lab)-lp(rl,lab))-(lp(cl,lab)-lp(rl,lab)))
            va.append((d>0).float().mean().item())
    print(f"  Ep{ep+1}: Loss={np.mean(ls):.4f} TrainPref={100*np.mean(pa):.1f}% ValPref={100*np.mean(va):.1f}%")

dc,de,dn=do_eval(dm,el,BS)
print(f"\n{'='*50}")
print(f"STEP 2: SFT={se:.2f}% → DPO={de:.2f}% (Δ={de-se:+.2f}pp)")
print(f"  {pc} pairs, {len(ti)}/{len(vi)} split")
print(f"{'='*50}")
torch.save(dm.state_dict(),"/kaggle/working/ckpts/dpo/dpo.pt")
del dm,rm,sft,so,el,pl; clean()
print(f"⏱️ DPO: {(time.time()-t0)/60:.1f} min")

---
# STEP 3: Generalization (Sudoku→Maze)

In [ ]:
from collections import deque

def build_maze(out, nt, ne, sz=6):
    for split,n in [("train",nt),("test",ne)]:
        res={k:[] for k in ["inputs","labels","puzzle_identifiers","puzzle_indices","group_indices"]}
        res["puzzle_indices"].append(0); res["group_indices"].append(0); g=0
        for _ in range(n*10):
            if g>=n: break
            grid=np.zeros((sz,sz),dtype=int)
            for r in range(sz):
                for c in range(sz):
                    if random.random()<0.25: grid[r,c]=1
            grid[0,0]=2; grid[sz-1,sz-1]=3
            q=deque([(0,0)]); vis={(0,0)}; par={}
            while q:
                r,c=q.popleft()
                if (r,c)==(sz-1,sz-1): break
                for dr,dc in [(0,1),(0,-1),(1,0),(-1,0)]:
                    nr,nc=r+dr,c+dc
                    if 0<=nr<sz and 0<=nc<sz and (nr,nc) not in vis and grid[nr,nc]!=1:
                        vis.add((nr,nc)); par[(nr,nc)]=(r,c); q.append((nr,nc))
            if (sz-1,sz-1) not in par: continue
            sol=grid.copy(); p=(sz-1,sz-1)
            while p!=(0,0):
                if sol[p]==0: sol[p]=4
                p=par[p]
            res["inputs"].append(grid.flatten()+1); res["labels"].append(sol.flatten()+1)
            res["puzzle_indices"].append(g+1); res["puzzle_identifiers"].append(0); res["group_indices"].append(g+1)
            g+=1
        res={k:np.array(v,dtype=np.int32) for k,v in res.items()}
        meta=PuzzleDatasetMetadata(seq_len=sz*sz,vocab_size=6,pad_id=0,ignore_label_id=0,
            blank_identifier_id=1,num_puzzle_identifiers=1,total_groups=g,mean_puzzle_examples=1,sets=["all"])
        d=os.path.join(out,split); os.makedirs(d,exist_ok=True)
        with open(os.path.join(d,"dataset.json"),"w") as f: json.dump(meta.model_dump(),f)
        for k,v in res.items(): np.save(os.path.join(d,f"all__{k}.npy"),v)
        with open(os.path.join(out,"identifiers.json"),"w") as f: json.dump(["<blank>"],f)
        print(f"  maze/{split}: {g}")

build_maze("data/maze-6x6",200,150)

t0=time.time()
cfg=CFGS["ACT"]
tl,els,meta=mkload(DATA,BS,42,train_epochs=EP)
gm,go=mkmodel(cfg["a"],cfg["l"],cfg["lt"],cfg["o"],meta,BS,42)
gm=do_train(gm,go,tl,EP,BS,LR,meta)
sc2,se2,_=do_eval(gm,els,BS)
print(f"Sudoku: Cell={sc2:.2f}% Exact={se2:.2f}%")

try:
    _,elm,_=mkload("data/maze-6x6",BS,42,ep=1)
    mc,me,_=do_eval(gm,elm,BS)
    print(f"Maze:   Cell={mc:.2f}% Exact={me:.2f}%")
except Exception as e:
    print(f"Maze eval error: {e}"); mc=me=0.0

print(f"\n{'='*50}\nSTEP 3: Sudoku={se2:.2f}% → Maze={me:.2f}% (gap={se2-me:.2f}pp)\n{'='*50}")
del gm,go,tl,els; clean()
print(f"⏱️ Step 3: {(time.time()-t0)/60:.1f} min")

---
# STEP 4: Layer Ablation (3 configs)

In [ ]:
t0=time.time()
LAYERS={
    "Shallow": dict(H_layers=2,L_layers=1,H_cycles=1,L_cycles=1,halt_max_steps=1,halt_exploration_prob=0.0),
    "Standard": dict(H_layers=4,L_layers=4,H_cycles=2,L_cycles=2,halt_max_steps=8,halt_exploration_prob=0.1),
    "Deep": dict(H_layers=8,L_layers=8,H_cycles=2,L_cycles=2,halt_max_steps=8,halt_exploration_prob=0.1),
}
abl={}
for name,lcfg in LAYERS.items():
    print(f"\n{'='*50}\n  {name}: H={lcfg['H_layers']} L={lcfg['L_layers']} cyc={lcfg['H_cycles']} halt={lcfg['halt_max_steps']}\n{'='*50}")
    over={**BASE,**lcfg}
    tl,el,meta=mkload(DATA,BS,42,train_epochs=EP)
    m,opts=mkmodel("hrm.hrm_act_v1@HierarchicalReasoningModel_ACTV1","losses@ACTLossHead","stablemax_cross_entropy",over,meta,BS,42)
    np_=sum(p.numel() for p in m.parameters())
    m=do_train(m,opts,tl,EP,BS,LR,meta)
    c,e,n=do_eval(m,el,BS)
    abl[name]=(c,e,np_)
    print(f"  ✅ Cell={c:.2f}% | Exact={e:.2f}% | {np_:,} params")
    d="/kaggle/working/ckpts/ablation"; os.makedirs(d,exist_ok=True)
    torch.save(m.state_dict(),f"{d}/{name}.pt")
    del m,opts,tl,el; clean()

print(f"\n⏱️ Step 4: {(time.time()-t0)/60:.1f} min")
print(f"\n{'Config':<12} {'Cell':>8} {'Exact':>8} {'Params':>12}")
print("-"*43)
for n,(c,e,p) in abl.items(): print(f"{n:<12} {c:>7.2f}% {e:>7.2f}% {p:>11,}")

---
# Final Summary

In [ ]:
print("#"*60)
print("#  NHRT FURTHER STEPS — RESULTS")
print("#"*60)

print("\n── Step 1: Seed Variance ──")
for a,r in seed_results.items():
    v=[r[s][1] for s in SEEDS if s in r]
    if v: print(f"  {a}: {np.mean(v):.2f}±{np.std(v):.2f}% ({len(v)} seeds)")

print(f"\n── Step 2: DPO ──")
print(f"  SFT={se:.2f}% → DPO={de:.2f}% (Δ={de-se:+.2f}pp, {pc} pairs)")

print(f"\n── Step 3: Generalization ──")
print(f"  Sudoku={se2:.2f}% → Maze={me:.2f}% (gap={se2-me:.2f}pp)")

print(f"\n── Step 4: Layer Ablation ──")
for n,(c,e,p) in abl.items(): print(f"  {n}: {e:.2f}% exact ({p:,} params)")

print(f"\nCheckpoints: /kaggle/working/ckpts/")
print("#"*60)

In [ ]:
# McNemar utility
from scipy.stats import binom_test
def mcnemar(cA,cB):
    b=((~cA)&cB).sum(); c_=(cA&(~cB)).sum(); n=b+c_
    if n==0: return "tie",1.0,"no discordant pairs"
    p=binom_test(min(b,c_),n,0.5)
    return ("B" if b>c_ else "A"), p, "sig" if p<0.05 else "ns"
print("mcnemar(correct_A, correct_B) loaded.")